# Dual-encoder FDR with InstaNovo top-k candidates

Pipeline:
1. Run InstaNovo beam search on each spectrum -> top-k candidate peptide sequences.
2. For each candidate, build a reversed-sequence decoy (drop if it collides with any target candidate for the same spectrum).
3. Encode the spectrum once with the spectrum encoder; encode every target+decoy peptide with the peptide encoder.
4. Score each (spectrum, peptide) pair by cosine similarity.
5. Per spectrum, take best-target score and best-decoy score.
6. Pool and run target-decoy FDR (q-value) over the best-target scores; decoys provide the null.

## Before running
- Copy the cells defining `SpectrumEncoder`, `PeptideEncoder`, `AA_VOCAB`, `MAX_PEPTIDE_LEN`, `preprocess_peptide`, and `preprocess_spectrum` from `DUAL_Encoder_v3_FIXED (2).ipynb` into a sibling file `dual_encoder_modules.py`. Then this notebook imports them cleanly.
- Set the paths in the **Config** cell.

## Imports

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

## Config

In [ ]:
# --- Paths ---
SPECTRA_PATH         = Path(r"C:/Users/tasne/Desktop/InstaSearch Project/InstaNovo/sample_data/spectra.mgf")
DUAL_ENCODER_CKPT    = Path(r"C:/Users/tasne/Desktop/InstaSearch Project/Data - Copy/dual_encoder.pt")  # TODO: point to your checkpoint
DUAL_ENCODER_MODULE  = Path(r"C:/Users/tasne/Desktop/InstaSearch Project")  # parent of dual_encoder_modules.py
OUT_DIR              = Path(r"C:/Users/tasne/Desktop/InstaSearch Project/Data - Copy/dualenc_fdr_out")
OUT_DIR.mkdir(exist_ok=True, parents=True)

# --- Inference / pipeline ---
INSTANOVO_MODEL = "instanovo-v1.2.0"
TOP_K           = 5         # beam width / number of candidates per spectrum
BATCH_SIZE      = 16        # InstaNovo inference batch size
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

# --- Decoy strategy ---
DECOY_STRATEGY  = "reverse"  # "reverse" or "shuffle"
DROP_DECOY_COLLISIONS = True  # drop decoys that match any target candidate of the same spectrum

# --- Cosine similarity / FDR ---
FDR_THRESHOLD   = 0.01      # 1% FDR
RANDOM_SEED     = 0
rng = np.random.default_rng(RANDOM_SEED)

print("DEVICE:", DEVICE)

## Dual-encoder model classes

Imported from `dual_encoder_modules.py` (created by copying the cells from `DUAL_Encoder_v3_FIXED (2).ipynb`). The expected interface is:

- `AA_VOCAB: dict[str, int]` with `<PAD>` at index 0 and the standard 20 amino acids.
- `MAX_PEPTIDE_LEN: int`
- `preprocess_peptide(seq: str) -> np.ndarray[int64, MAX_PEPTIDE_LEN]`
- `preprocess_spectrum(mz: np.ndarray, intensity: np.ndarray, precursor_mz: float, precursor_charge: int) -> tuple[np.ndarray, np.ndarray]` returning `(peaks_padded, precursors)`
- `SpectrumEncoder()` callable as `model_spec(x, precursors) -> [B, D]`
- `PeptideEncoder()`  callable as `model_pep(tokens) -> [B, D]`

In [ ]:
if str(DUAL_ENCODER_MODULE) not in sys.path:
    sys.path.insert(0, str(DUAL_ENCODER_MODULE))

from dual_encoder_modules import (
    SpectrumEncoder, PeptideEncoder,
    AA_VOCAB, MAX_PEPTIDE_LEN,
    preprocess_peptide, preprocess_spectrum,
)

model_spec = SpectrumEncoder().to(DEVICE)
model_pep  = PeptideEncoder().to(DEVICE)

ckpt = torch.load(DUAL_ENCODER_CKPT, map_location=DEVICE)
# Expected keys: 'model_spec', 'model_pep'. Adjust if your checkpoint uses different names.
model_spec.load_state_dict(ckpt["model_spec"])
model_pep.load_state_dict(ckpt["model_pep"])
model_spec.eval(); model_pep.eval()
print("Dual-encoder loaded.")

## Step 1: InstaNovo top-k candidates per spectrum

In [ ]:
from instanovo.transformer.model import InstaNovo
from instanovo.inference.beam_search import BeamSearchDecoder
from instanovo.utils.data_handler import SpectrumDataFrame
from instanovo.transformer.data import SpectrumDataset, collate_batch
from torch.utils.data import DataLoader

in_model, in_config = InstaNovo.from_pretrained(INSTANOVO_MODEL)
in_model = in_model.to(DEVICE).eval()
decoder = BeamSearchDecoder(model=in_model)

sdf = SpectrumDataFrame.from_mgf(str(SPECTRA_PATH))
ds = SpectrumDataset(
    sdf,
    in_model.residue_set,
    n_peaks=in_config.get("n_peaks", 200),
    return_str=True,
)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
print(f"Loaded {len(sdf)} spectra.")

In [ ]:
candidates = []  # list of dicts: spectrum_idx, candidate_seq, candidate_log_prob, rank
spectrum_meta = []  # list of dicts per spectrum (mz, intensity, precursor_mz, precursor_charge, scan)

spec_idx = 0
with torch.no_grad():
    for batch in tqdm(loader, desc="InstaNovo beam search"):
        spectra, precursors, spectra_mask, _, _ = batch
        spectra      = spectra.to(DEVICE)
        precursors   = precursors.to(DEVICE)
        spectra_mask = spectra_mask.to(DEVICE)

        out = decoder.decode(
            spectra=spectra,
            precursors=precursors,
            beam_size=TOP_K,
            max_length=MAX_PEPTIDE_LEN,
            return_beam=True,
        )

        # `out` is a list of length B; each entry is a list of beam dicts with 'predictions' and 'log_probability'
        for i, beams in enumerate(out):
            for rank, beam in enumerate(beams[:TOP_K]):
                seq = "".join(beam["predictions"]) if isinstance(beam["predictions"], (list, tuple)) else str(beam["predictions"])
                candidates.append({
                    "spectrum_idx": spec_idx + i,
                    "rank": rank,
                    "candidate_seq": seq,
                    "candidate_log_prob": float(beam.get("log_probability", beam.get("sequence_log_probability", np.nan))),
                })
        spec_idx += spectra.shape[0]

cand_df = pd.DataFrame(candidates)
print(cand_df.shape)
cand_df.head(10)

## Step 2: Reversed-peptide decoys per candidate

In [ ]:
AA_PATTERN = "".join(c for c in AA_VOCAB if c.isalpha() and c.isupper())

def strip_modifications(seq: str) -> str:
    """Drop UNIMOD/modification annotations like 'C[UNIMOD:4]' or 'M(ox)' so the AA encoder sees clean residues."""
    import re
    seq = re.sub(r"\[[^\]]*\]", "", seq)
    seq = re.sub(r"\([^)]*\)", "", seq)
    return "".join(c for c in seq if c in AA_PATTERN)

def make_decoy(seq: str, strategy: str = DECOY_STRATEGY) -> str:
    s = strip_modifications(seq)
    if strategy == "reverse":
        return s[::-1]
    elif strategy == "shuffle":
        arr = list(s)
        rng.shuffle(arr)
        return "".join(arr)
    raise ValueError(strategy)

cand_df["target_seq"] = cand_df["candidate_seq"].apply(strip_modifications)
cand_df["decoy_seq"]  = cand_df["target_seq"].apply(make_decoy)

if DROP_DECOY_COLLISIONS:
    targets_per_spec = cand_df.groupby("spectrum_idx")["target_seq"].agg(set).to_dict()
    cand_df["decoy_collides"] = cand_df.apply(
        lambda r: r["decoy_seq"] in targets_per_spec[r["spectrum_idx"]], axis=1
    )
    print(f"Dropping {cand_df['decoy_collides'].sum()} colliding decoys (will be excluded from scoring).")
else:
    cand_df["decoy_collides"] = False

cand_df.head()

## Step 3: Encode spectra (one embedding per spectrum)

In [ ]:
# Re-iterate the SpectrumDataFrame to grab raw peaks/precursor info, run them through your dual-encoder spectrum preprocessor.
spec_records = []
for i, row in enumerate(sdf):
    spec_records.append({
        "spectrum_idx": i,
        "mz":            np.asarray(row["mz_array"], dtype=np.float32),
        "intensity":     np.asarray(row["intensity_array"], dtype=np.float32),
        "precursor_mz":  float(row["precursor_mz"]),
        "precursor_charge": int(row["precursor_charge"]),
    })

def encode_spectra(records, batch_size=64):
    embs = []
    with torch.no_grad():
        for start in tqdm(range(0, len(records), batch_size), desc="Encoding spectra"):
            batch = records[start:start+batch_size]
            peaks_list, prec_list = [], []
            for r in batch:
                peaks, prec = preprocess_spectrum(r["mz"], r["intensity"], r["precursor_mz"], r["precursor_charge"])
                peaks_list.append(peaks); prec_list.append(prec)
            x          = torch.as_tensor(np.stack(peaks_list)).to(DEVICE)
            precursors = torch.as_tensor(np.stack(prec_list)).to(DEVICE)
            z = model_spec(x, precursors)
            z = F.normalize(z, dim=-1)
            embs.append(z.cpu())
    return torch.cat(embs, dim=0)

z_spec_all = encode_spectra(spec_records)
print("Spectrum embeddings:", z_spec_all.shape)

## Step 4: Encode all candidate + decoy peptides; cosine similarity

In [ ]:
def encode_peptides(seqs, batch_size=512):
    embs = []
    with torch.no_grad():
        for start in tqdm(range(0, len(seqs), batch_size), desc="Encoding peptides"):
            chunk = seqs[start:start+batch_size]
            tokens = np.stack([preprocess_peptide(s) for s in chunk])
            tokens = torch.as_tensor(tokens, dtype=torch.long).to(DEVICE)
            z = model_pep(tokens)
            z = F.normalize(z, dim=-1)
            embs.append(z.cpu())
    return torch.cat(embs, dim=0)

target_embs = encode_peptides(cand_df["target_seq"].tolist())
decoy_embs  = encode_peptides(cand_df["decoy_seq"].tolist())

spec_idx_t = torch.as_tensor(cand_df["spectrum_idx"].to_numpy(), dtype=torch.long)
z_spec_per_cand = z_spec_all[spec_idx_t]

cand_df["target_score"] = (z_spec_per_cand * target_embs).sum(dim=-1).numpy()
cand_df["decoy_score"]  = (z_spec_per_cand * decoy_embs ).sum(dim=-1).numpy()
cand_df.loc[cand_df["decoy_collides"], "decoy_score"] = np.nan
cand_df.head()

## Step 5: Best target / best decoy per spectrum

In [ ]:
best_target_idx = cand_df.groupby("spectrum_idx")["target_score"].idxmax()
best_target = cand_df.loc[best_target_idx, ["spectrum_idx", "target_seq", "target_score", "candidate_log_prob", "rank"]].rename(
    columns={"target_seq": "best_seq", "target_score": "best_score", "rank": "best_rank"}
).reset_index(drop=True)

decoy_valid = cand_df.dropna(subset=["decoy_score"])
best_decoy_idx = decoy_valid.groupby("spectrum_idx")["decoy_score"].idxmax()
best_decoy = decoy_valid.loc[best_decoy_idx, ["spectrum_idx", "decoy_seq", "decoy_score"]].rename(
    columns={"decoy_seq": "best_decoy_seq", "decoy_score": "best_decoy_score"}
).reset_index(drop=True)

per_spec = best_target.merge(best_decoy, on="spectrum_idx", how="left")
per_spec.head()

## Step 6: Target-decoy FDR / q-values

Standard TDA: pool best-target and best-decoy scores (one each per spectrum), label them target vs decoy, sort by score descending. At each threshold s,

FDR(s) = #decoys with score >= s / #targets with score >= s

q-value = monotone running minimum of FDR from below.

In [ ]:
pool = pd.concat([
    pd.DataFrame({"spectrum_idx": per_spec["spectrum_idx"], "score": per_spec["best_score"], "is_decoy": False, "seq": per_spec["best_seq"]}),
    pd.DataFrame({"spectrum_idx": per_spec["spectrum_idx"], "score": per_spec["best_decoy_score"], "is_decoy": True,  "seq": per_spec["best_decoy_seq"]}),
], ignore_index=True).dropna(subset=["score"]).reset_index(drop=True)

pool = pool.sort_values("score", ascending=False).reset_index(drop=True)
pool["cum_decoys"]  = pool["is_decoy"].cumsum()
pool["cum_targets"] = (~pool["is_decoy"]).cumsum()
pool["fdr"]    = pool["cum_decoys"] / pool["cum_targets"].clip(lower=1)
pool["qvalue"] = pool["fdr"][::-1].cummin()[::-1]

n_accepted = int((pool["qvalue"] <= FDR_THRESHOLD).sum())
n_target_accepted = int(((pool["qvalue"] <= FDR_THRESHOLD) & (~pool["is_decoy"])).sum())
print(f"At {FDR_THRESHOLD:.0%} FDR: {n_target_accepted} target PSMs accepted (of {(~pool['is_decoy']).sum()} total target spectra).")

pool.to_csv(OUT_DIR / "dualenc_pooled_scores.csv", index=False)
per_spec.to_csv(OUT_DIR / "dualenc_per_spectrum.csv", index=False)
cand_df.to_csv(OUT_DIR / "dualenc_candidates.csv", index=False)

## Step 7: Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(per_spec["best_score"].dropna(),       bins=80, alpha=0.7, label="target",  color="#769FB6")
axes[0].hist(per_spec["best_decoy_score"].dropna(), bins=80, alpha=0.7, label="decoy",   color="#9DBBAE")
axes[0].set_xlabel("cosine similarity (best per spectrum)"); axes[0].set_ylabel("count"); axes[0].legend(); axes[0].set_title("Score distribution")

axes[1].plot(pool["score"], pool["qvalue"], lw=1)
axes[1].axhline(FDR_THRESHOLD, color="red", ls="--", lw=1)
axes[1].set_xlabel("score threshold"); axes[1].set_ylabel("q-value"); axes[1].set_title("q-value vs threshold")
axes[1].invert_xaxis()

qsweep = np.linspace(0.001, 0.1, 50)
n_accept = [(pool["qvalue"] <= q).sum() - 2 * (pool[(pool["qvalue"] <= q) & pool["is_decoy"]].shape[0]) for q in qsweep]
# rough net-target curve: targets accepted minus decoys accepted
n_targets_accept = [((pool["qvalue"] <= q) & (~pool["is_decoy"])).sum() for q in qsweep]
axes[2].plot(qsweep, n_targets_accept, marker="o", ms=3)
axes[2].axvline(FDR_THRESHOLD, color="red", ls="--", lw=1)
axes[2].set_xlabel("q-value cutoff"); axes[2].set_ylabel("# target PSMs accepted"); axes[2].set_title("PSMs vs FDR")

plt.tight_layout(); plt.show()

## Notes / things to verify

- **Beam-search return shape.** `BeamSearchDecoder.decode(..., return_beam=True)` is expected to return `list[list[dict]]`. If the dict keys differ in your InstaNovo version (e.g. `sequence_log_probability` vs `log_probability`), adjust the parser in the beam-search loop.
- **Spectrum preprocessor parity.** `preprocess_spectrum` *must* match the function the dual encoder was trained with — otherwise the spectrum embeddings are out of distribution. If the training notebook normalized intensities, capped peak count, etc., copy that exact function.
- **Modification handling.** `strip_modifications` removes UNIMOD/parenthetical mods so the peptide encoder sees plain residues. If your dual encoder was trained with modifications represented in `AA_VOCAB`, replace `strip_modifications` with whatever tokenizer the encoder expects — otherwise you will systematically degrade target scores.
- **Decoy length / composition.** Reverse-decoys preserve length and AA composition perfectly, which is what we want for an unbiased null. Shuffle is an alternative if your peptide encoder is sensitive to N-/C-terminal residue identity (since reversal moves K/R to the N-term).
- **Top-k vs single best.** Picking `argmax` over the top-k targets is what the analogous "best PSM per scan" rule does in TDA. If you also want a per-rank FDR (e.g. how often the rank-1 InstaNovo guess is correct under the dual-encoder rescoring), keep `cand_df` and group by `rank`.